# Part 5 · 00 Setup and input check

This notebook runs once before anything else in part 5. It:

1. records the environment (Python and package versions, for the reproducibility checklist);
2. checks that every input from parts 1–2 and 3–4 is where part 5 expects it, and says which mode `cache_raw.py` will run in;
3. runs the unit tests of the cleaning on small built documents (no data, no network).

Part 5 needs neither WRDS nor any login: the WRDS data it uses (CRSP names, I/B/E/S link, the event table) were already pulled or built by parts 3–4.

**Before running:** clone the repository and open this notebook inside it. Set `TEAM_DIR` if the private data live outside the checkout.

**Folder layout**

```
5_data cleaning/
  code/     settings.py (paths), cache_raw, clean_release (+ tests), entities,
            token_stats, dedup, model_cache, splits, manifest
  cache/    raw EDGAR bytes (raw/raw_YYYY.zip + raw_manifest.parquet) and model outputs
            (model_outputs.sqlite)
  data/     built tables keyed by accession: clean/, chunks/ (pieces per model), entities,
            token_stats, dedup_flags, splits_draft, holdout_ids.txt once frozen
  results/  tables for the report (no row-level data)
  logs/     environment records
  manifest.json  every output with its rows, sha256 and the script that wrote it
```

**Inputs read, never changed:** from `files csv/` (or `1_2nov/nov2_data` when present), `extraction_report.csv` (the document `fetch_releases.py` chose for each filing) and `event_table.csv` (company names); `scripts/fetch_releases.py`; from `3_4_surprise_returns`, `data/events_final.parquet` (event keys, reaction dates, splits, sample flags), the release clock `data/clock_provisional.csv`, the trading calendar, `cache/crsp_names.parquet`, `cache/link_ibes_crsp.parquet` and `SPLITS` in `code/common.py`.

**Read by the next parts:** `data/clean/clean_YYYY.parquet` (parts 6–8), `data/chunks/<model>/` (part 6), `cache/model_outputs.sqlite` (written by part 6 through `ModelCache`, read by parts 7–9), `data/entities.parquet` (masking) and `data/holdout_ids.txt` (parts 7–8). See `README.md`.

In [ ]:
%pip install -q pandas numpy pyarrow beautifulsoup4 lxml nltk datasketch tiktoken transformers
# %pip installs into the environment of this kernel. If it upgraded a package that
# was already loaded, restart the kernel and continue from the next cell.

In [ ]:
import os, subprocess, sys
from pathlib import Path

# Find the checkout from the repository root or any notebook subfolder.
_start = Path(os.environ.get("S08_ROOT", Path.cwd())).expanduser().resolve()
REPO_ROOT = next((p for p in (_start, *_start.parents)
                  if (p / "scripts").is_dir()
                  and (p / "3_4_surprise_returns" / "code").is_dir()), None)
if REPO_ROOT is None:
    raise FileNotFoundError("Open this notebook inside the cloned S08 repository, "
                            "or set S08_ROOT to its path.")
# Code always comes from this checkout. Private data (cache/, data/) live under
# TEAM_DIR: the checkout itself unless TEAM_DIR is set.
TEAM_DIR = Path(os.environ.get("TEAM_DIR", REPO_ROOT)).expanduser().resolve()
STEP5_DIR = Path(os.environ.get("STEP5_DIR", TEAM_DIR / "5_data cleaning")).expanduser().resolve()
CODE = REPO_ROOT / "5_data cleaning" / "code"

for sub in ["cache", "data", "results", "logs"]:
    (STEP5_DIR / sub).mkdir(parents=True, exist_ok=True)

# settings.py, and every script started below, read the roots from these.
os.environ["S08_ROOT"] = str(REPO_ROOT)   # rerunning this cell after the chdir still finds it
os.environ["TEAM_DIR"] = str(TEAM_DIR)
os.environ["STEP5_DIR"] = str(STEP5_DIR)
if str(CODE) not in sys.path:
    sys.path.insert(0, str(CODE))
os.chdir(STEP5_DIR)       # relative artifacts stay in the private data folder


def run_script(name, *args):
    """Run code/<name> with this kernel's Python, show its output here as it comes,
    and stop the notebook if it fails."""
    cmd = [sys.executable, "-u", str(CODE / name), *map(str, args)]
    env = {**os.environ, "PYTHONIOENCODING": "utf-8"}
    with subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, env=env,
                          text=True, encoding="utf-8", errors="replace") as p:
        for line in p.stdout:
            print(line, end="")
    if p.returncode:
        raise subprocess.CalledProcessError(p.returncode, cmd)

missing = [f for f in ["settings.py", "cache_raw.py", "clean_release.py", "test_clean_release.py",
                                 "entities.py", "token_stats.py", "dedup.py", "model_cache.py",
                                 "splits.py", "manifest.py"] if not (CODE / f).exists()]
assert not missing, f"Missing code in {CODE}: {missing}"
print("repository:   ", REPO_ROOT)
print("data root:    ", TEAM_DIR)
print("step 5 folder:", STEP5_DIR)
print("code files:   ", sorted(os.listdir(CODE)))

In [ ]:
# Environment record: goes into logs/ and, later, the README.
import json, platform, datetime as dt
import importlib.metadata as md

env = {"run_at_utc": dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds"),
       "python": platform.python_version()}
for pkg in ["pandas", "numpy", "pyarrow", "beautifulsoup4", "lxml", "nltk",
            "datasketch", "tiktoken", "transformers"]:
    try:
        env[pkg] = md.version(pkg)
    except md.PackageNotFoundError:
        env[pkg] = "not installed"
(STEP5_DIR / "logs" / f"env_{dt.date.today()}.json").write_text(json.dumps(env, indent=2))
print(json.dumps(env, indent=2))

## Inputs from the other parts

Every file part 5 reads, with its size. `extraction_report.csv` names the document `fetch_releases.py` chose for each filing (`accession`, `cik`, `filename`), so `cache_raw.py` makes one request per document (3 to 4 hours); without it, it would read each index page again (7 to 8 hours). `events_final.parquet` is written by part 4 (`04_sample`); until it exists, the splits are drafted from the release clock alone and cannot be frozen.

In [ ]:
import pandas as pd
import settings as S

print("inputs from parts 1–2:", S.NOV2_DATA)

inputs = {
    "extraction report (parts 1–2)": S.EXTRACTION_REPORT,
    "event table (parts 1–2)": S.EVENT_TABLE,
    "fetch_releases.py (scripts)": S.NOV2_CODE / "fetch_releases.py",
    "events_final (part 4)": S.EVENTS_FINAL,
    "release clock (3_4)": S.CLOCK,
    "trading calendar (3_4)": S.CALENDAR,
    "CRSP names (3_4 cache)": S.CRSP_NAMES,
    "I/B/E/S-CRSP link (3_4 cache)": S.IBES_LINK,
    "common.py with SPLITS (3_4)": S.REPO_ROOT / "3_4_surprise_returns" / "code" / "common.py",
}
for label, p in inputs.items():
    print(f"  {'ok ' if p.exists() else 'MISSING'}  {label:<38} "
          f"{p.stat().st_size if p.exists() else 0:>12,}")

report = S.fetch_report()
print()
print("cache_raw.py mode:", f"report ({report.name})" if report else
      "index (no report naming the documents: each index page is read again)")
if report:
    rep = pd.read_csv(report, usecols=["accession", "source"], dtype=str, keep_default_na=False)
    has_doc = ~rep.source.isin(["none", "index fetch failed", ""])
    print(f"  {len(rep):,} filings, {int(has_doc.sum()):,} with a document to cache")

if S.CLOCK.exists():
    clk = pd.read_csv(S.CLOCK, usecols=["accession", "reaction_date", "status"], dtype=str)
    print(f"\nrelease clock: {len(clk):,} rows, {clk.accession.nunique():,} accessions, "
          f"{clk.reaction_date.notna().sum():,} with a reaction date")
    print(clk.status.value_counts().to_string())
if S.EVENTS_FINAL.exists():
    ev = pd.read_parquet(S.EVENTS_FINAL, columns=["accession", "split", "in_text_sample", "in_primary"])
    t = ev[ev.in_text_sample.astype(bool)]
    print(f"\nevents_final: {len(ev):,} events, {int(ev.accession.notna().sum()):,} with a filing; "
          f"text sample {len(t):,}, primary {int(ev.in_primary.astype(bool).sum()):,}")
    print(t.split.value_counts().to_string())
else:
    print("\nno events_final.parquet yet (part 4): the splits will be drafted from the clock "
          "and cannot be frozen")
print("\nsplits from common.py:", S.common_3_4().SPLITS)

## Unit tests of the cleaning

Small documents built like the kinds met in the sample (a recent HTML release, an old release laid out in a table, an old `.txt` filing in SGML, an iXBRL 8-K body, repeated content, an earnings release against a monthly traffic report). Every line should start with `ok`.

In [ ]:
run_script("test_clean_release.py")